# Notebook para calcular los umbrales de seguridad

In [12]:
import pandas as pd

Aplica a las lecturas meteorológicas los umbrales de seguridad especificados en Proyecto_Aruba.pdf y `docs/umbrales-seguridad-aruba.md`: la **alerta de déficit eólico (caída de viento)** y la **alerta de sobrecarga por consumo (pico de demanda)**. A partir de ambas se obtiene el estado global de la red.

Definimos las constantes de los umbrales de seguridad: capacidad nominal del parque eólico, velocidades del viento (crítico, moderado y *cut-out*) y límite de consumo.

In [13]:
WIND_CAPACITY_KW = 30000.0     # Capacidad nominal del Parque Eólico Vader Piet (30 MW)
WIND_CRITICAL_KMH = 15.0      # < 15 km/h: generación < 15% de la capacidad nominal
WIND_MODERATE_KMH = 25.0      # < 25 km/h: generación entre 15% y 40%
WIND_CUT_OUT_KMH = 65.0       # > 65 km/h: desconexión por cut-out
CONSUMPTION_OVERLOAD_KW = 35000.0

Leemos el csv de lecturas y hacemos una copia del DataFrame para no modificar el original. Sobre `df` se van añadiendo las columnas de alerta en las celdas siguientes.

In [14]:
df_readings = pd.read_csv('../data/processed/aruba_reading_filtered_by_station_distance.csv')
df = df_readings.copy()

Definición de umbrales numéricos basados en el 40% (12,000 kW) y 15% (4,500 kW) de la capacidad nominal

In [15]:
threshold_40_pct = WIND_CAPACITY_KW * 0.40
threshold_15_pct = WIND_CAPACITY_KW * 0.15

## **Alerta de Déficit Eólico (Caída de Viento)** — Parque Vader Piet, 30 MW nominales. Calculamos las condiciones de déficit:
- Si existe `wind_power_kw` (potencia real):
  - Moderado: < 40% de capacidad nominal (12,000 kW) → respaldo solar / BESS
  - Crítico: < 15% de capacidad nominal (4,500 kW) → despacho térmico 100%
- Si no existe, se clasifica por `wind_speed_kmh` (equivalencias de la doc, sección 2.3):
  - Moderado: < 25 km/h (~15–40% de capacidad)
  - Crítico: < 15 km/h (< 15% de capacidad)

In [16]:
if 'wind_power_kw' in df.columns:
    is_critical_deficit = df['wind_power_kw'] < threshold_15_pct
    is_moderate_deficit = df['wind_power_kw'] < threshold_40_pct
else:
    is_critical_deficit = df['wind_speed_kmh'] < WIND_CRITICAL_KMH
    is_moderate_deficit = df['wind_speed_kmh'] < WIND_MODERATE_KMH

Completamos la alerta eólica con la condición de viento extremo y asignamos la etiqueta `wind_alert`:
- Crítico por *cut-out*: > 65 km/h (las turbinas frenan por protección)
- Prioridad de la etiqueta: `CRITICAL_CUT_OUT` → `CRITICAL_THERMAL_BACKUP_100%` → `MODERATE_SOLAR_BATTERY_BACKUP` → `NORMAL`

**Cómo se asigna la etiqueta (`.loc`)**: las condiciones (`is_cut_out`, `is_critical_deficit`, `is_moderate_deficit`) son Series de `True`/`False`, una por fila. Primero se marca toda la columna como `NORMAL` y luego `df.loc[condición, 'wind_alert'] = etiqueta` sobrescribe solo las filas donde la condición es `True`, todas a la vez y sin recorrer el DataFrame con un bucle.

Una misma lectura puede cumplir varias condiciones a la vez (por ejemplo, 10 km/h está por debajo de 25 y de 15 km/h). Como cada asignación pisa a la anterior, **se escriben de menos a más grave**: así la última, la más grave, es la que se queda. Si se invirtiera el orden, una lectura crítica acabaría marcada como moderada.

In [17]:
if 'wind_speed_kmh' in df.columns:
    is_cut_out = df['wind_speed_kmh'] > WIND_CUT_OUT_KMH
else:
    is_cut_out = pd.Series(False, index=df.index)

Asigna una etiqueta de alerta eólica: se parte de 'NORMAL' y cada .loc sobrescribe las filas que cumplen su condición. Gana la última asignación, por eso el orden va de menos a más grave

In [18]:
df['wind_alert'] = 'NORMAL'
df.loc[is_moderate_deficit, 'wind_alert'] = 'MODERATE_SOLAR_BATTERY_BACKUP'
df.loc[is_critical_deficit, 'wind_alert'] = 'CRITICAL_THERMAL_BACKUP_100%'
df.loc[is_cut_out, 'wind_alert'] = 'CRITICAL_CUT_OUT'


## **Alerta de Sobrecarga por Consumo (Pico de Demanda)**:
- Crítico: `predicted_consumption_kw` > 35,000 kW
- Si la columna no existe, se muestra un aviso y la alerta queda en `False`

Comprueba si el consumo predicho supera el límite de seguridad de 35,000 kW

In [19]:
if 'predicted_consumption_kw' in df.columns:
    df['consumption_overload_alert'] = df['predicted_consumption_kw'] > CONSUMPTION_OVERLOAD_KW
else:
    print("[WARN] Falta 'predicted_consumption_kw': no se evalúa la alerta de sobrecarga por consumo")
    df['consumption_overload_alert'] = False

[WARN] Falta 'predicted_consumption_kw': no se evalúa la alerta de sobrecarga por consumo


Por último, `grid_state` combina ambas alertas: cualquier alerta crítica (eólica o de consumo) → `CRITICAL`; alerta eólica moderada → `MODERATE`; resto → `OK`. Se asigna con el mismo patrón de `.loc` (de menos a más grave).

In [20]:
is_critical = df['wind_alert'].str.startswith('CRITICAL') | df['consumption_overload_alert']
is_moderate = df['wind_alert'].str.startswith('MODERATE')

Mismo patrón que en wind_alert: de menos a más grave, la última asignación gana

In [21]:
df['grid_state'] = '[OK] Stable Operation'
df.loc[is_moderate, 'grid_state'] = '[MODERATE] Solar/BESS Backup Required'
df.loc[is_critical, 'grid_state'] = '[CRITICAL] Thermal Dispatch Required'

Mostramos cuántas lecturas hay en cada nivel de alerta

In [22]:
print(df['wind_alert'].value_counts())
print(df['consumption_overload_alert'].value_counts())
print(df['grid_state'].value_counts())

wind_alert
MODERATE_SOLAR_BATTERY_BACKUP    876193
NORMAL                           473276
CRITICAL_THERMAL_BACKUP_100%       1539
Name: count, dtype: int64
consumption_overload_alert
False    1351008
Name: count, dtype: int64
grid_state
[MODERATE] Solar/BESS Backup Required    876193
[OK] Stable Operation                    473276
[CRITICAL] Thermal Dispatch Required       1539
Name: count, dtype: int64
